# Pandas — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока.


In [ ]:
# Setup
%matplotlib inline

import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tempfile
import warnings
from pathlib import Path
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


## Постановка задачи `(intro)`


Сквозной сценарий на **Titanic (OpenML)**: EDA в pandas, индексы и маски, groupby/transform, кодирование, merge, Parquet, sklearn Pipeline без утечки.


## Загрузка и первичный EDA `(eda)`


In [ ]:
warnings.filterwarnings("ignore")


sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"].astype(str)
df["Embarked"] = df["embarked"].astype(str)

print(f"Объектов: {len(df)}")
df.info()
print(df.describe(include="all").T.head(8))
print("Пропуски:\n", df[["Age", "Fare", "Embarked"]].isnull().sum())


## Аналитика и кардинальность `(eda)`


In [ ]:
print(df["Sex"].value_counts())
print("nunique name:", df["name"].nunique(), "— почти ID, в модель не берём")
corr = df[["Survived", "Pclass", "Age", "Fare"]].corr()
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("corr() перед моделью")
plt.tight_layout()
plt.show()


## Очистка: пропуски и признаки `(example)`


In [ ]:
df_clean = df.copy()
df_clean["Age"] = df_clean.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean["Fare"] = df_clean["Fare"].fillna(df_clean.groupby("Pclass")["Fare"].transform("median"))

df_clean["is_expensive"] = np.where(df_clean["Fare"] > df_clean["Fare"].median(), 1, 0)
upper = df_clean["Fare"].quantile(0.99)
df_clean["Fare_clip"] = df_clean["Fare"].clip(upper=upper)
df_clean["fare_dev"] = df_clean["Fare"] - df_clean.groupby("Pclass")["Fare"].transform("mean")
print("Пропуски Age:", df_clean["Age"].isna().sum())
df_clean[["Fare", "Fare_clip", "fare_dev", "is_expensive"]].head()


## groupby, MultiIndex и merge `(viz)`


In [ ]:
summary = df_clean.groupby(["Pclass", "Sex"]).agg(
    n=("Survived", "size"),
    survival=("Survived", "mean"),
    avg_fare=("Fare", "mean"),
).round(3)
print(summary)

ports = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "port_name": ["Southampton", "Cherbourg", "Queenstown"],
})
df_ports = pd.merge(df_clean, ports, on="Embarked", how="left")
df_ports.groupby("port_name")["Survived"].mean().sort_values(ascending=False)


## Кодирование и Parquet `(example)`


In [ ]:
df_model = pd.get_dummies(df_clean, columns=["Sex", "Embarked"], drop_first=True)
feature_cols = ["Pclass", "Age", "Fare_clip", "is_expensive", "fare_dev"] + [
    c for c in df_model.columns if c.startswith("Sex_") or c.startswith("Embarked_")
]
df_model = df_model[feature_cols + ["Survived"]].reset_index(drop=True)

pq_path = Path(tempfile.mkdtemp()) / "titanic_clean.parquet"
try:
    df_model.to_parquet(pq_path, index=False)
    df_loaded = pd.read_parquet(pq_path)
    print("Загружено из Parquet:", df_loaded.shape)
except ImportError:
    csv_path = pq_path.with_suffix(".csv")
    df_model.to_csv(csv_path, index=False)
    df_loaded = pd.read_csv(csv_path)
    print("pyarrow нет — сохранено в CSV:", df_loaded.shape)


## train_test_split и Pipeline `(example)`


In [ ]:
X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(classification_report(y_test, pred, digits=3))

X_np = pipe.named_steps["scaler"].transform(pipe.named_steps["imputer"].transform(X_test))
print("numpy dtype:", X_np.dtype, "NaN:", np.isnan(X_np).any())


## Память и чек-лист `(summary)`


In [ ]:
del df_ports, summary
gc.collect()

print("Чек-лист:")
print("  info/describe/value_counts/corr — EDA")
print("  transform для контекстных признаков")
print("  get_dummies + Parquet для типов")
print("  split до fit-статистик; Pipeline для imputer/scaler")
print("  to_numpy перед sklearn; del + gc перед тяжёлой моделью")


1. EDA: `info`, `describe`, `value_counts`, `corr`.
2. Индексы: `reset_index` после фильтров и перед concat.
3. Пропуски и признаки — осмысленно; clip/qcut/np.where.
4. `groupby` agg + `transform`; merge по уникальному ключу.
5. OHE; промежуточные данные — Parquet.
6. `train_test_split` → Pipeline → numpy → метрики.
7. `del` + `gc.collect()` в Jupyter.
